In [25]:
import pandas as pd
import numpy as np
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [26]:
data = pd.read_csv("../data/query.csv")

X = data["query"]
y = data["intent"]

In [27]:
data.sample(5)

,query,intent
408,request to remove umc case imposed on me durin...,Security_and_Safety
431,company interview schedule not visible on plac...,Placements
179,Immigration support contact?,International_Affair
106,Discipline issue on campus,Security_and_Safety
101,Complaint about eve teasing,Security_and_Safety


In [28]:
data.shape

(558, 2)

In [29]:
print(data['intent'].value_counts())

intent
Residential_Services                    69
Examination                             67
Academics                               67
Division_of_Research_and_Development    55
Library                                 54
Placements                              54
Fees_and_Account                        48
IT                                      40
Security_and_Safety                     38
Admissions                              34
International_Affair                    32
Name: count, dtype: int64


In [30]:
data[data["intent"]=="Library"]

,query,intent
82,library system not allowing me to reserve book...,Library
112,What are library timings?,Library
113,How to reserve a book?,Library
114,Check book availability?,Library
115,Library fine payment process?,Library
116,How to get library no‑dues?,Library
117,Access online journals?,Library
118,Plagiarism software access?,Library
119,Library discipline rules?,Library
120,Digital books access?,Library


In [31]:
import re
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

data['query'] = data['query'].apply(preprocess_text)
data.head()

,query,intent
0,how can i reset my university email password,IT
1,campus wifi is not working properly,IT
2,internet speed is very slow on campus,IT
3,i forgot my ums password,IT
4,ums portal login error is showing,IT


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
X_train_embeddings = sbert_model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

X_test_embeddings = sbert_model.encode(
    X_test.tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [34]:
clf = LogisticRegression(max_iter=1000)

clf.fit(X_train_emb, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [35]:
y_pred = clf.predict(X_test_embeddings)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))

Model Accuracy: 0.8482142857142857

Classification Report:

                                      precision    recall  f1-score   support

                           Academics       0.64      0.78      0.70         9
                          Admissions       0.67      0.50      0.57         8
Division_of_Research_and_Development       0.85      0.92      0.88        12
                         Examination       0.62      1.00      0.77         5
                    Fees_and_Account       0.92      0.85      0.88        13
                                  IT       0.83      0.91      0.87        11
                International_Affair       1.00      0.80      0.89         5
                             Library       1.00      0.90      0.95        10
                          Placements       1.00      0.79      0.88        14
                Residential_Services       0.90      0.90      0.90        20
                 Security_and_Safety       0.83      1.00      0.91         5

  

In [36]:
joblib.dump(clf, "../models/intent_classifier.pkl")

['../models/intent_classifier.pkl']

In [39]:
query = ["my wifi is not working"]

embedding = sbert_model.encode(query)

prediction = clf.predict(embedding)

probabilities = clf.predict_proba(embedding)

print("Predicted intent:", prediction)

print("Probabilities:", probabilities)

Predicted intent: ['IT']
Probabilities: [[0.03657357 0.01631819 0.03383006 0.04144276 0.02139794 0.46380544
  0.02090064 0.06886379 0.05771549 0.20491417 0.03423794]]
